# DEXTRA · 01 — Download external data (ERA5 + FIRMS → Hugging Face)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/01_download_external.ipynb)

Downloads everything the DEXTRA pipeline needs **except CPCB station data**, and stores it in your private Hugging Face dataset repo:

| Section | Source | What | Typical time |
|---|---|---|---|
| A | Open-Meteo (ERA5) | Hourly surface weather at the 15 cities, 2019–2025, raw ERA5 grid-cell values | ~10 min |
| A2 | Copernicus CDS (ERA5) | Boundary-layer height for Jan–Jun 2024, which Open-Meteo is missing | done once |
| B | NASA FIRMS | MODIS active-fire detections over India, 2019–2025 | ~10–20 min |
| C | Copernicus CDS (ERA5) | Temperature at 850–950 hPa at 00/03 UTC, for the inversion index | **hours per year** (CDS queue) |
| D | — | Completeness and cross-source consistency checks | ~5 min |

CPCB station data can't be scripted (the CPCB repository is behind a CAPTCHA), so it is downloaded in a browser separately and uploaded to the same repo.

### Before you run: add these 4 Colab Secrets
Click the **🔑 key icon** in the left sidebar → **Add new secret**, and switch **Notebook access** on for each:

| Secret name | Value |
|---|---|
| `HF_TOKEN` | Hugging Face token with **write** access to your dataset repo |
| `HF_DATASET_REPO` | Your dataset repo id, e.g. `your-username/dextra-aqi-india` |
| `CDSAPI_KEY` | Your Copernicus CDS API key (CDS profile page → *API key*) |
| `FIRMS_MAP_KEY` | Your NASA FIRMS MAP_KEY |

Never paste keys into a cell: anything in a cell is saved with the notebook and can end up on GitHub.

### Runtime
This notebook only downloads, so it doesn't need a GPU. It runs fine on a **T4** runtime, but a **CPU** runtime (*Runtime → Change runtime type → CPU*) saves your free GPU hours for training.

### Safe to re-run — and expect to
Every finished piece is uploaded to Hugging Face immediately, and every section skips what is already there.

Section C is slow, so it runs last, because ECMWF asks users to submit CDS requests **one at a time** (accounts that submit in parallel can be suspended), and each request can wait hours in their queue. The notebook records each CDS request ID on Hugging Face the moment it is submitted, so if Colab disconnects, **Runtime → Run all** re-attaches to the waiting request instead of joining the back of the queue again.


## 0 · Settings

In [ ]:
# Leave HF_DATASET_REPO empty to read it from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

RUN_METADATA = True          # city table + Diwali dates
RUN_ERA5_SURFACE = True      # Section A
RUN_BLH_GAP_FILL = True      # Section A2
RUN_FIRMS = True             # Section B
RUN_ERA5_PRESSURE = True     # Section C (slow)

CDS_POLL_SECONDS = 60        # how often to ask CDS whether a queued request is done

## 1 · Secrets
Read from Colab Secrets. Values are never printed — only whether each one was found.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    """Read a Colab secret without ever printing its value."""
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
CDSAPI_KEY = colab_secret("CDSAPI_KEY", required=RUN_ERA5_PRESSURE or RUN_BLH_GAP_FILL)
FIRMS_MAP_KEY = colab_secret("FIRMS_MAP_KEY", required=RUN_FIRMS)

print("Dataset repo:", HF_DATASET_REPO)
print("Secrets found:", {"HF_TOKEN": bool(HF_TOKEN), "CDSAPI_KEY": bool(CDSAPI_KEY), "FIRMS_MAP_KEY": bool(FIRMS_MAP_KEY)})

## 2 · Install packages and get the DEXTRA code
If you pulled new code into a session that already ran, use *Runtime → Restart session* first: Python keeps the old modules loaded otherwise.

In [ ]:
%pip install -q "cdsapi>=0.7.7" netCDF4 "huggingface_hub>=0.23.0"

In [ ]:
import os
import subprocess
import sys
import time

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import pandas as pd
from src import config
from src.data import calendar_events, download_era5, download_firms
from src.utils.hf_store import HFStore

store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
YEARS = list(range(int(config.DATA_START_DATE[:4]), int(config.DATA_END_DATE[:4]) + 1))
CITY_SLUG = {city: city.lower() for city in config.TARGET_CITIES}
SURFACE_PATH = "raw/era5_surface_grid/{}.parquet"


def cds_runner():
    """One CDS client for the whole notebook; requests go through it one at a time."""
    from ecmwf.datastores import Client
    return download_era5.CdsJobRunner(Client(url=config.CDS_URL, key=CDSAPI_KEY), store, poll_seconds=CDS_POLL_SECONDS)


print(f"Connected to {HF_DATASET_REPO}: {len(store.files())} files already there")
print(f"Years: {YEARS[0]}–{YEARS[-1]}, cities: {len(config.TARGET_CITIES)}")

## 3 · Metadata
The city table and the Diwali dates the feature pipeline uses. Small, so it is re-uploaded every run (unchanged files make no new commit).

In [ ]:
if RUN_METADATA:
    cities = pd.DataFrame([{"city": city, **meta} for city, meta in config.TARGET_CITIES.items()])
    for path, df in {
        "meta/cities.csv": cities,
        "meta/diwali_dates.csv": calendar_events.diwali_table(),
    }.items():
        store.upload_df(df, path, f"Metadata: {path}")
        print(f"↑ {path}: {len(df)} rows")

## A · ERA5 surface weather (Open-Meteo, raw grid-cell values)
Hourly boundary-layer height, 2 m temperature and dewpoint, surface pressure, precipitation and 10 m wind (m/s) at the ERA5 grid cell nearest each city, in UTC. One file per city: `raw/era5_surface_grid/<city>.parquet`.

Values are the **raw ERA5 grid cell** (`elevation=nan`). By default Open-Meteo rescales temperature and pressure to a 90 m terrain model, which moved Pune by 0.9 °C and 16 hPa, so the values would no longer line up with the pressure-level temperatures from CDS. The older `raw/era5_surface/` folder holds those rescaled values and is no longer used; you can delete it from Hugging Face whenever you like.

In [ ]:
if RUN_ERA5_SURFACE:
    for city in config.TARGET_CITIES:
        path = SURFACE_PATH.format(CITY_SLUG[city])
        if store.exists(path):
            print(f"✓ {city}: already on Hugging Face, skipped")
            continue
        df = download_era5.fetch_surface_city(city, YEARS)
        store.upload_df(df, path, f"ERA5 surface, raw grid cell (Open-Meteo) for {city}, {YEARS[0]}-{YEARS[-1]}")
        other_missing = df.drop(columns=["city", "time_utc", "blh", "grid_lat", "grid_lon", "grid_elevation"]).isna().mean().max()
        print(f"↑ {city}: {len(df):,} hourly rows, grid cell ({df.grid_lat.iloc[0]}, {df.grid_lon.iloc[0]}, "
              f"{df.grid_elevation.iloc[0]:.0f} m), BLH missing {df.blh.isna().mean():.1%} (patched in A2), "
              f"other variables missing ≤ {other_missing:.1%}")

## A2 · Boundary-layer height gap fill (Copernicus CDS)
Open-Meteo's ERA5 archive is missing boundary-layer height for **every hour of Jan–Jun 2024**, in every city, which is exactly DEXTRA's validation period. This section finds the missing months, fetches them from CDS together with one neighbouring month both sources have, and stores the patch separately (`raw/era5_blh_patch/<year>.parquet`) so the raw Open-Meteo files stay untouched. Section D shows how closely the two sources agree on the shared month.

Once a patch exists this section is skipped without reading anything.

In [ ]:
if RUN_BLH_GAP_FILL:
    existing_patch = store.files("raw/era5_blh_patch/")
    if existing_patch:
        print("✓ Patch already on Hugging Face, skipped:", existing_patch)
    else:
        surface = pd.concat([store.read_df(SURFACE_PATH.format(CITY_SLUG[c])) for c in config.TARGET_CITIES], ignore_index=True)
        gaps = download_era5.blh_gap_months(surface)
        print("Months with missing boundary-layer height:", gaps or "none")
        runner = cds_runner()
        for year, months in gaps.items():
            check_month = download_era5.overlap_month(months)
            print(f"… {year}: one CDS request for months {months} + overlap month {check_month}")
            patch = download_era5.download_blh_months(runner, year, months + [check_month], "/content/era5_tmp")
            store.upload_df(patch, f"raw/era5_blh_patch/{year}.parquet", f"ERA5 boundary-layer height patch (CDS) {year}")
            print(f"↑ {year}: {len(patch):,} hourly rows")

## B · NASA FIRMS fire detections
MODIS Collection 6.1 detections over India and its neighbours (the Pakistani Punjab and Bangladesh lie inside the 500 km transport radius). Quality-controlled standard-processing data is used wherever it exists; the last few months, which it doesn't cover yet, come from near-real-time data and are marked in the `source` column. One file per year: `raw/firms/modis_<year>.parquet`.

A FIRMS key allows 5000 transactions per 10 minutes, and each 5-day request over India uses several, so the download checks the key's quota as it goes and pauses when it runs low. If FIRMS still refuses a year, the cell reports why and moves on, so the slow CDS section below still starts; run this cell again later to fetch that year.

In [ ]:
if RUN_FIRMS:
    try:
        availability = download_firms.data_availability(FIRMS_MAP_KEY)
        sp_end = download_firms.archive_end_date(availability, download_firms.SP_SOURCE)
    except RuntimeError as err:
        print("Could not read FIRMS data availability, using standard processing throughout:", err)
        sp_end = None
    print("MODIS standard-processing archive ends:", sp_end)

    firms_failed = []
    for year in YEARS:
        path = f"raw/firms/modis_{year}.parquet"
        if store.exists(path):
            print(f"✓ {year}: already on Hugging Face, skipped")
            continue
        try:
            df = download_firms.fetch_year(FIRMS_MAP_KEY, year, sp_end)
        except download_firms.FirmsError as err:
            firms_failed.append(year)
            print(f"✗ {year}: {err}")
            continue
        store.upload_df(df, path, f"NASA FIRMS MODIS fire detections {year}")
        sources = df["source"].value_counts().to_dict() if len(df) else {}
        print(f"↑ {year}: {len(df):,} detections {sources}")

    if firms_failed:
        print()
        print(f"⚠ FIRMS years not downloaded yet: {firms_failed}. The notebook carries on to section C; "
              "run this cell again later to fetch them (finished years are skipped).")

## C · Pressure-level temperature for the inversion index (Copernicus CDS) — slow, runs last
Temperature at 850, 875, 900, 925 and 950 hPa at 00 and 03 UTC (05:30 and 08:30 IST, when inversions are strongest). Each city later uses the level ~75 hPa above its own surface; a fixed 925 hPa is underground in Bengaluru. The 2 m temperature it is compared with comes from Section A. Section D can be run on its own at any time while this waits. One file per year: `raw/era5_pressure/<year>.parquet`.

For each year the notebook, in order:
1. skips it if `raw/era5_pressure/<year>.parquet` exists;
2. reuses an earlier `raw/era5_inversion/<year>.parquet` if one exists (no CDS request);
3. re-attaches to a matching request already on your CDS account, whether recorded by this notebook or submitted by an earlier version of it;
4. otherwise submits **one** request and waits for it before moving to the next year.

**This is the slow part:** a request can wait several hours in the CDS queue. If Colab disconnects, run again and it picks up the same request.

In [ ]:
if RUN_ERA5_PRESSURE:
    runner = cds_runner()
    for year in YEARS:
        path = f"raw/era5_pressure/{year}.parquet"
        if store.exists(path):
            print(f"✓ {year}: already on Hugging Face, skipped")
            continue

        earlier = f"raw/era5_inversion/{year}.parquet"
        if store.exists(earlier):
            df = download_era5.pressure_from_inversion_file(store.read_df(earlier))
            store.upload_df(df, path, f"ERA5 pressure-level temperature {year} (from earlier inversion file)")
            print(f"↑ {year}: {len(df):,} rows taken from {earlier}, no CDS request needed")
            continue

        print(f"… {year}: CDS request (one at a time, as ECMWF requires; can wait hours in their queue)")
        started = time.time()
        df = download_era5.download_pressure_year(runner, year, "/content/era5_tmp")
        store.upload_df(df, path, f"ERA5 pressure-level temperature (CDS) {year}")
        print(f"↑ {year}: {len(df):,} rows, {(time.time() - started) / 3600:.1f} h in total")

## D · Checks
### D1 · Completeness
Every file in use, with the row count each source should produce.

In [ ]:
store.refresh()
expected_hours = len(pd.date_range(f"{YEARS[0]}-01-01", f"{YEARS[-1]}-12-31 23:00", freq="h"))
n_cities = len(config.TARGET_CITIES)

def expected_rows(path):
    if path.startswith("raw/era5_surface_grid/"):
        return expected_hours
    if path.startswith("raw/era5_pressure/"):
        year = int(path.rsplit("/", 1)[1][:4])
        return n_cities * len(pd.date_range(f"{year}-01-01", f"{year}-12-31")) * len(config.INVERSION_HOURS_UTC)
    return None

IN_USE = ("raw/era5_surface_grid/", "raw/era5_blh_patch/", "raw/era5_pressure/", "raw/firms/")
rows, frames = [], {}
for path in [p for p in store.files("raw/") if p.startswith(IN_USE)]:
    frames[path] = df = store.read_df(path)
    expected = expected_rows(path)
    ok = (len(df) == expected) if expected else (len(df) > 0)
    rows.append({"file": path, "rows": len(df), "expected": expected or "> 0", "ok": "✓" if ok else "✗"})

summary = pd.DataFrame(rows)
display(summary)
print(f"{(summary.ok == '✓').sum()} of {len(summary)} files look complete")
missing_years = [y for y in YEARS if f"raw/era5_pressure/{y}.parquet" not in frames]
print("Pressure-level years still to come:", missing_years or "none")
missing_firms = [y for y in YEARS if f"raw/firms/modis_{y}.parquet" not in frames]
print("FIRMS years still to come:", missing_firms or "none")
superseded = [p for p in store.files("raw/") if p.startswith(("raw/era5_surface/", "raw/era5_inversion/"))]
print(f"{len(superseded)} superseded files (raw/era5_surface/, raw/era5_inversion/) are kept but not used.")

### D2 · Do the sources agree?
Both checks compare two copies of the **same ERA5 reanalysis**, so differences should be tiny and correlations close to 1.
- **Boundary-layer height:** CDS patch vs Open-Meteo on the overlap month.
- **2 m temperature:** CDS (from the earlier inversion files) vs Open-Meteo raw grid values at 00/03 UTC. Agreement here is what justifies taking 2 m temperature for the inversion index from Open-Meteo instead of a second CDS request per year.

In [ ]:
surface = pd.concat([frames[SURFACE_PATH.format(CITY_SLUG[c])] for c in config.TARGET_CITIES
                     if SURFACE_PATH.format(CITY_SLUG[c]) in frames], ignore_index=True)

patch_paths = [p for p in frames if p.startswith("raw/era5_blh_patch/")]
if patch_paths and len(surface):
    blh = download_era5.blh_agreement(surface, pd.concat([frames[p] for p in patch_paths]))
    print("Boundary-layer height, CDS patch vs Open-Meteo (overlap month):")
    display(blh.round(3))

old_inversion = [p for p in store.files("raw/era5_inversion/")]
if old_inversion and len(surface):
    cds_t2m = pd.concat([store.read_df(p)[["city", "time_utc", "temp2m"]] for p in old_inversion], ignore_index=True)
    t2m = download_era5.series_agreement(surface, cds_t2m, "temp2m")
    print("2 m temperature (°C), Open-Meteo raw grid vs CDS:")
    display(t2m.round(3))
    worst = t2m["mean_abs_diff"].max()
    print("✓ Sources agree" if worst < 0.3 else f"⚠ Largest mean difference {worst:.2f} °C; tell me before building the dataset.")

### D3 · Inversion level per city
The pressure level each city's inversion index will use: the one closest to 75 hPa above its mean grid-cell surface pressure.

In [ ]:
if len(surface):
    levels = download_era5.inversion_levels(surface)
    mean_sp = surface.groupby("city")["surface_pressure"].mean().round(1)
    display(pd.DataFrame({"mean surface pressure (hPa)": mean_sp, "inversion level (hPa)": pd.Series(levels)}))

## Next
1. **CPCB station data:** downloaded in a browser (CAPTCHA), then uploaded to `raw/cpcb/` in the same repo.
2. **`02_build_dataset.ipynb`:** hourly → daily with CPCB's rules, city medians, AQI, merge with ERA5 + FIRMS, train/val/test splits.